***Classification model***
**For classification model, I will predict whether the tip will be given**
I will use the following features:
- passenger count
- trip distance
- pick up hour
- pick up day of the week
- trip fare
Since dataset is balance, based on the tip applied column, no further preprocessing is needed.

I will use the random forest classifier;

**Metrics**
- F1-score
- Accuracy
- Precision
- Recall
- Overall confusion matrix

In [9]:
import sys
from pathlib import Path
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import ParameterGrid
from sklearn.metrics import f1_score, accuracy_score, precision_score, recall_score, confusion_matrix
import pandas as pd
import pickle

sys.path.insert(0, str(Path.cwd().parent))

from HW1.data_preprocessing import load_and_process, train_val_test_split

In [10]:
repo_root = Path.cwd() if (Path.cwd() / "data").is_dir() else Path.cwd().parent
data = load_and_process("data/green_tripdata_2021-01.parquet", from_dvc=True, repo=repo_root)
data.head()

,passenger_count,trip_distance,fare_amount,pickup_hour,pickup_day_of_week,tip_applied
0,1.0,1.01,5.5,0,4,0
1,1.0,2.53,10.0,0,4,1
2,1.0,1.12,6.0,0,4,1
3,1.0,1.99,8.0,23,3,0
7,6.0,0.45,3.5,0,4,1


In [11]:
train, val, test = train_val_test_split(data, random_state = 42)

In [12]:
print(f"train size: {train.shape}, val size: {val.shape}, test size: {test.shape}")

train size: (26473, 6), val size: (5673, 6), test size: (5673, 6)


In [13]:
# Feature columns (per specification) and target
FEATURE_COLUMNS = [
    "passenger_count",
    "trip_distance",
    "pickup_hour",
    "pickup_day_of_week",
    "fare_amount",
]
TARGET_COLUMN = "tip_applied"

X_train = train[FEATURE_COLUMNS]
X_val = val[FEATURE_COLUMNS]
X_test = test[FEATURE_COLUMNS]

y_train = train[TARGET_COLUMN]
y_val = val[TARGET_COLUMN]
y_test = test[TARGET_COLUMN]

In [14]:
X_train.head()

,passenger_count,trip_distance,pickup_hour,pickup_day_of_week,fare_amount
16514,1.0,1.73,18,2,7.5
12476,1.0,1.75,9,0,9.5
14584,2.0,5.19,14,1,19.0
37241,1.0,1.48,11,4,8.5
21960,1.0,0.47,0,0,4.0


In [15]:
param_grid = {
    "n_estimators": [50, 100, 200, 300],
    "max_depth": [3, 6, 10, 20],
    "random_state": [42],
}

In [16]:
results = []
for params in ParameterGrid(param_grid):
    model = RandomForestClassifier(**params)
    model.fit(X_train, y_train)
    y_train_pred = model.predict(X_train)
    y_val_pred = model.predict(X_val)
    results.append({
        **params,
        "train_f1": f1_score(y_train, y_train_pred),
        "train_accuracy": accuracy_score(y_train, y_train_pred),
        "val_f1": f1_score(y_val, y_val_pred),
        "val_accuracy": accuracy_score(y_val, y_val_pred),
    })

# Store results and best model (by val F1)
best_idx = max(range(len(results)), key=lambda i: results[i]["val_f1"])
best_params = {k: v for k, v in results[best_idx].items() if k in param_grid}
best_f1_score = results[best_idx]["val_f1"]
best_model = RandomForestClassifier(**best_params).fit(X_train, y_train)

In [17]:
results_df = pd.DataFrame(results).sort_values("val_f1", ascending=False)
results_df[["n_estimators", "max_depth", "val_f1", "val_accuracy", "train_f1", "train_accuracy"]]

,n_estimators,max_depth,val_f1,val_accuracy,train_f1,train_accuracy
5,100,6,0.623522,0.573418,0.635023,0.587844
6,200,6,0.623486,0.572713,0.634577,0.587618
7,300,6,0.623134,0.572713,0.633862,0.586824
10,200,10,0.622686,0.579588,0.680160,0.646017
11,300,10,0.621835,0.578706,0.680791,0.647150
0,50,3,0.621514,0.564604,0.624047,0.566162
4,50,6,0.619577,0.571831,0.633839,0.589053
3,300,3,0.619398,0.562842,0.624407,0.566691
2,200,3,0.619289,0.563723,0.624024,0.567031
9,100,10,0.618373,0.577472,0.678987,0.645677


In [18]:
print("Best parameters:", best_params)
print("Best validation F1:", best_f1_score)
y_val_pred = best_model.predict(X_val)
print("Validation F1:", f1_score(y_val, y_val_pred))
print("Validation accuracy:", accuracy_score(y_val, y_val_pred))

Best parameters: {'max_depth': 6, 'n_estimators': 100, 'random_state': 42}
Best validation F1: 0.6235220908525202
Validation F1: 0.6235220908525202
Validation accuracy: 0.573417944650097


In [19]:
# Evaluate best model on test set
y_test_pred = best_model.predict(X_test)

print("Test set metrics:")
print("F1-score:", f1_score(y_test, y_test_pred))
print("Accuracy:", accuracy_score(y_test, y_test_pred))
print("Precision:", precision_score(y_test, y_test_pred))
print("Recall:", recall_score(y_test, y_test_pred))
print("\nConfusion matrix:")
print(confusion_matrix(y_test, y_test_pred))

Test set metrics:
F1-score: 0.6076742364917777
Accuracy: 0.558434690639873
Precision: 0.5481774512574173
Recall: 0.6816584680252986

Confusion matrix:
[[1228 1599]
 [ 906 1940]]


In [22]:
with open('models/model_pkl_v1', 'wb') as files:
    pickle.dump(best_model, files)